# Parameters and a water state


**Evidence: numerical verification.** Compute liquid water at 298.15 K and
100,000 Pa, then return to the same density to check pressure closure.

The canonical input is `held-cameretti-sadowski-2008/4`. Its water coefficients
come from Held, Cameretti and Sadowski (2008), Table 1
([paper](https://doi.org/10.1016/j.fluid.2008.06.010)); the packet retains the
NIST molar mass and source-specific input conventions. Read `packet.toml`
beside the parameter file for its exact provenance and use basis.


## Read the inputs

Use a repository checkout with the environment described in the installation
guide. The wheel alone does not contain these data. All density, association,
equilibrium and parameter-fit iterations below run in the native Engine.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from epcsaft import (
    Parameters,
    Pure,
    ThermochemistryStatus,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "ARCHITECTURE.yaml").is_file()
)
PACKETS = ROOT / "data" / "packets"


## Resolve pressure to density

`Pure` selects one component from the packet. A pressure request asks the Engine
for the liquid density branch; a density request evaluates that state directly.
The numerical criterion is an absolute pressure difference below 0.001 Pa.
Molar density times molar mass gives mass density in kg/m³.


In [2]:
parameters = Parameters.from_json(
    PACKETS / "held-cameretti-sadowski-2008/4/parameters/parameters.json",
    components=["water"],
)
model = Pure(parameters)
state = model.state(298.15, P=100000.0, phase="liquid")
density_state = model.state(298.15, rho=state.molar_density)
assert abs(density_state.pressure - 100000.0) < 0.001
values = dict(
    rho_mol_m3=state.molar_density,
    rho_kg_m3=state.molar_density * parameters.molar_masses[0],
    closure_Pa=density_state.pressure - 100000.0,
    total_enthalpy=state.total_enthalpy,
    thermal_cause=state.thermochemistry_status.name,
)
assert state.total_enthalpy is None
assert state.thermochemistry_status == ThermochemistryStatus.MissingRecord
display(
    pd.DataFrame([values]).rename(
        columns={
            "rho_mol_m3": "Molar density / mol m^-3",
            "rho_kg_m3": "Mass density / kg m^-3",
            "closure_Pa": "Pressure closure / Pa",
        }
    )
)


,Molar density / mol m^-3,Mass density / kg m^-3,Pressure closure / Pa,total_enthalpy,thermal_cause
0,55342.16982,997.004685,0.000002,None,MissingRecord


## Meaning and limits


The density is a source-parameter model prediction. No observed water density
is used here, so pressure closure verifies the calculation rather than physical
accuracy. Missing ideal thermal inputs leave total enthalpy unavailable with
`MissingRecord`; valid density and residual properties remain usable.
